# 다이캐스팅 품질 예측 프로젝트 (End-to-End, XGBoost)

이 노트북은 **비전공자도 이해할 수 있도록** 설명을 최대한 자세히 적어둔 완성형 분석 파이프라인입니다.

## 목표
1. 데이터를 불러오고(컬럼 구조 이해)
2. 불량(Defects)을 **표면/구조 그룹**으로 묶어 **Multi-label 타겟(y)**을 만든 뒤
3. 공정(Process) + 센서(Sensor) 변수로 **XGBoost 모델**을 학습합니다.
4. **데이터 누출(leakage)** 위험이 있는 컬럼을 자동으로 제거하여 **신뢰도 있는 모델**을 만듭니다.
5. Feature importance / (선택) SHAP / “불량 최소 조건” 탐색까지 이어집니다.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report, roc_auc_score, average_precision_score

# XGBoost (설치가 안 되어 있으면 아래 주석을 해제 후 실행)
# !pip -q install xgboost
from xgboost import XGBClassifier

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

plt.rcParams['font.family'] = 'Malgun Gothic' 
plt.rcParams['axes.unicode_minus'] = False

## 1. 데이터 로드

이 프로젝트에서는 `product_type_1.csv`를 사용합니다.

이 CSV는 **2줄 헤더(MultiIndex)** 구조일 수 있습니다.
- 상위 레벨: process / sensor / defects
- 하위 레벨: velocity_1, bubble_1 등

그래서 `header=[0,1]`로 먼저 읽어보고,
- MultiIndex면 그대로 사용
- 아니면 일반 컬럼(flat)으로 처리합니다.


In [ ]:
DATA_PATH = "product_type_1.csv"

try:
    df1 = pd.read_csv(DATA_PATH, header=[0, 1])
    multi = True
except Exception:
    df1 = pd.read_csv(DATA_PATH)
    multi = False

print("MultiIndex header:", multi)
print("Shape:", df1.shape)

if multi:
    print("Top-level columns:", df1.columns.levels[0].tolist())
else:
    print("Columns sample:", df1.columns[:15].tolist())

df1.head()


## 2. process / sensor / defects 블록 분리

- MultiIndex라면 `df1['process']` 처럼 접근 가능
- flat 컬럼이라면 `process_...`, `sensor_...`, `defects_...` 접두어로 찾아 분리

아래 코드는 두 경우를 모두 안전하게 처리합니다.


In [ ]:
def is_multiindex_columns(df: pd.DataFrame) -> bool:
    return isinstance(df.columns, pd.MultiIndex)

def get_block(df: pd.DataFrame, block_name: str) -> pd.DataFrame:
    if is_multiindex_columns(df):
        return df[block_name].copy()
    prefix = block_name.lower() + "_"
    cols = [c for c in df.columns if str(c).lower().startswith(prefix)]
    return df[cols].copy()

process_df = get_block(df1, "process")
sensor_df  = get_block(df1, "sensor")
defects_df = get_block(df1, "defects")

print("process_df:", process_df.shape)
print("sensor_df :", sensor_df.shape)
print("defects_df:", defects_df.shape)
print("\n[defects columns]")
print(defects_df.columns.tolist())


## 3. defect 그룹화 → multi-label 타겟(y) 만들기

우리가 예측할 y는 2개입니다.

- **표면**: dent/stain/exfoliation 중 하나라도 있으면 1
- **구조**: short_shot/bubble/deformation 중 하나라도 있으면 1

주의:
- defects 값이 0/1 뿐 아니라 2,3도 있을 수 있으므로
  - **0이면 양품**
  - **0초과면 불량(1로 통일)**
  로 이진화합니다.


In [ ]:
defects_numeric = defects_df.apply(pd.to_numeric, errors="coerce").fillna(0)
defects_bin = (defects_numeric > 0).astype(int)

defect_groups = {
    "표면": ["dent_1", "stain_1", "exfoliation_1", "exfoliation_2"],
    "구조": ["short_shot_1", "short_shot_2", "bubble_1", "bubble_2", "deformation_1", "deformation_2"],
}

def normalize_defect_colname(col):
    s = str(col).lower()
    return s.replace("defects_", "") if s.startswith("defects_") else s

map_to_real = {normalize_defect_colname(c): c for c in defects_bin.columns}

y = pd.DataFrame(index=df1.index)
for group_name, cols in defect_groups.items():
    real_cols = [map_to_real[c] for c in cols if c in map_to_real]
    if len(real_cols) == 0:
        raise KeyError(f"[{group_name}] 그룹에 해당하는 defects 컬럼을 찾지 못했습니다: {cols}")
    y[group_name] = defects_bin[real_cols].any(axis=1).astype(int)

print("[표면/구조 불량 비율(%)]")
display((y.mean() * 100).round(2).to_frame("rate_%"))
y.head()


## 4. X 만들기 + 데이터 누출(leakage) 방지

X는 **process + sensor**만 사용합니다.

그리고 아래는 “미래 예측에서 알 수 없거나”, “식별자/시간”이라서  
모델이 **공정 원인이 아니라 꼼수 패턴을 외우게** 만들 수 있습니다.

- process_id: 단순 식별번호
- process_shot: 생산 순서/카운터(시간 변수 성격)
- process_product_type: 이번 분석에서는 product_type_1만 사용 → 불필요/혼입 위험
- defect_flag_is_defect / is_defect: 이름부터 정답 힌트 가능

그래서 이런 컬럼을 **자동 탐지해서 제거**합니다.


In [ ]:
X = pd.concat([process_df, sensor_df], axis=1).copy()
X = X.drop(columns=["id","shot"], errors="ignore")
print("X shape (before drop):", X.shape)

def col_to_str(c):
    if isinstance(c, tuple):
        return "_".join([str(x) for x in c]).lower()
    return str(c).lower()

leak_keywords = [
    "process_id",
    "process_shot",
    "process_product_type",
    "defect_flag_is_defect",
    "is_defect",
]

leak_cols = [c for c in X.columns if any(k in col_to_str(c) for k in leak_keywords)]
if leak_cols:
    print("🚫 drop leakage cols:", leak_cols)
    X = X.drop(columns=leak_cols, errors="ignore")
else:
    print("✅ leakage 키워드 컬럼 없음")

defect_like = [c for c in X.columns if "defects" in col_to_str(c)]
if defect_like:
    print("🚫 drop defects-like cols from X:", defect_like)
    X = X.drop(columns=defect_like, errors="ignore")

print("X shape (after drop):", X.shape)


## 5. Train/Test 분리 (multi-label stratify)

y가 (표면, 구조) 2개 컬럼이므로 `stratify=y`를 그대로 쓰면 오류가 나거나 불안정할 수 있습니다.

그래서 (표면,구조) 조합을 문자열로 만들어 strata로 사용합니다.
- 00, 10, 01, 11 같은 형태


In [ ]:
strata = y.astype(str).agg("".join, axis=1)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=strata
)

print("X_train:", X_train.shape, "X_test:", X_test.shape)
print("y_train:", y_train.shape, "y_test:", y_test.shape)

print("\n[라벨 조합 분포(전체)]")
display(strata.value_counts(normalize=True).sort_index().to_frame("ratio"))


## 6. XGBoost 모델 학습 (표면/구조 각각 1개 모델)

Multi-label이지만, 타겟(표면/구조)마다 불량 비율이 달라서  
타겟별로 `scale_pos_weight = (양품 수 / 불량 수)`를 따로 적용합니다.

평가 지표:
- ROC-AUC
- PR-AUC (불균형에서 특히 중요)
- precision / recall / f1


In [ ]:
imputer = SimpleImputer(strategy="median")
X_train_imp = imputer.fit_transform(X_train)
X_test_imp  = imputer.transform(X_test)

models, preds, probas = {}, {}, {}

for t in y_train.columns:
    neg = int((y_train[t] == 0).sum())
    pos = int((y_train[t] == 1).sum())
    spw = float(neg / max(pos, 1))
    print(f"\n[{t}] scale_pos_weight = {spw:.3f} (neg={neg}, pos={pos})")

    model = XGBClassifier(
        n_estimators=800,
        max_depth=6,
        learning_rate=0.03,
        subsample=0.8,
        colsample_bytree=0.8,
        gamma=0.1,
        reg_lambda=1.0,
        scale_pos_weight=spw,
        eval_metric="logloss",
        random_state=42,
        n_jobs=-1
    )

    model.fit(X_train_imp, y_train[t])

    proba = model.predict_proba(X_test_imp)[:, 1]
    pred  = (proba >= 0.5).astype(int)

    models[t] = model
    probas[t] = proba
    preds[t]  = pred

    print(classification_report(y_test[t], pred, digits=4))
    roc = roc_auc_score(y_test[t], proba)
    pr  = average_precision_score(y_test[t], proba)
    print(f"ROC-AUC: {roc:.4f} | PR-AUC: {pr:.4f}")

y_pred  = pd.DataFrame(preds, index=y_test.index)
y_proba = pd.DataFrame(probas, index=y_test.index)

display(y_pred.head())
display(y_proba.head())


## 7. (선택) Threshold 튜닝

현재는 확률이 0.5 이상이면 불량이라고 판단합니다.

불량이 적은 데이터에서는 0.5가 너무 높아서 불량을 많이 놓칠 수 있으니,  
0.05~0.80 구간에서 threshold를 바꿔가며 precision/recall trade-off를 확인합니다.


In [ ]:
def evaluate_threshold(y_true, proba, threshold):
    pred = (proba >= threshold).astype(int)
    rep = classification_report(y_true, pred, output_dict=True, zero_division=0)
    return {
        "threshold": threshold,
        "precision_1": rep["1"]["precision"],
        "recall_1": rep["1"]["recall"],
        "f1_1": rep["1"]["f1-score"],
        "accuracy": rep["accuracy"]
    }

for t in y_test.columns:
    rows = []
    for th in np.linspace(0.05, 0.80, 16):
        rows.append(evaluate_threshold(y_test[t].values, y_proba[t].values, float(th)))
    df_th = pd.DataFrame(rows)

    print(f"\n=== [{t}] threshold sweep (recall 높은 순 TOP5) ===")
    display(df_th.sort_values("recall_1", ascending=False).head(5))

    plt.figure(figsize=(6,4))
    plt.plot(df_th["threshold"], df_th["precision_1"], marker="o", label="precision(불량=1)")
    plt.plot(df_th["threshold"], df_th["recall_1"], marker="o", label="recall(불량=1)")
    plt.title(f"{t}: Precision/Recall vs Threshold")
    plt.xlabel("threshold")
    plt.ylabel("score")
    plt.ylim(0, 1.0)
    plt.legend()
    plt.show()


## 8. Feature Importance

XGBoost는 각 변수의 중요도를 제공합니다.  
중요도가 높다고 “원인”이라고 단정할 수는 없지만,  
공정 점검/최적화 후보를 고르는 데 매우 유용합니다.


In [ ]:
for t, model in models.items():
    imp = pd.Series(model.feature_importances_, index=X.columns).sort_values(ascending=False)
    print(f"\n=== [{t}] TOP 15 Feature Importance ===")
    display(imp.head(15).to_frame("importance"))

    plt.figure(figsize=(8,5))
    topk = imp.head(15)[::-1]
    plt.barh(topk.index.astype(str), topk.values)
    plt.title(f"{t} - Top 15 Feature Importances (XGBoost)")
    plt.xlabel("importance")
    plt.show()


## 9. “불량 최소 조건” 탐색

중요도 TOP 변수들을 10분위(Decile)로 나눠 구간별 불량률을 계산합니다.

- 어떤 구간에서 불량률이 가장 낮은지 확인
- 공정 조건 최적화 후보를 좁히는 데 도움

주의: 이것은 인과가 아니라 패턴입니다.


In [ ]:
TOP_N = 5

for t, model in models.items():
    print(f"\n=== [{t}] 불량 최소 조건 탐색 (TOP {TOP_N}) ===")
    imp = pd.Series(model.feature_importances_, index=X.columns).sort_values(ascending=False)
    top_vars = imp.head(TOP_N).index.tolist()

    df_tmp = X_test.copy()
    df_tmp["y_true"] = y_test[t].values

    rows = []
    for v in top_vars:
        try:
            bins = pd.qcut(df_tmp[v], q=10, duplicates="drop")
            rate = df_tmp.groupby(bins)["y_true"].mean()
            rows.append({
                "var": str(v),
                "best_bin": str(rate.idxmin()),
                "best_defect_rate": float(rate.min())
            })
        except Exception as e:
            rows.append({
                "var": str(v),
                "best_bin": "N/A",
                "best_defect_rate": np.nan,
                "note": repr(e)
            })

    display(pd.DataFrame(rows))


## 10. leakage 제거 최종 확인

아래 결과에 `shot`, `id`, `is_defect`, `defects` 관련 컬럼이 남아 있으면  
다시 X에서 제거해야 합니다.


In [ ]:
[c for c in X.columns if "defect" in str(c).lower() or "shot" in str(c).lower() or "id" in str(c).lower()]